# 12.14 打字或說話，怎麼交給同一位聊天助手？


最容易檢查的第一個完整系統，是把語音當成另一個輸入入口。打字直接進聊天模型；說話先經語音辨識，得到逐字稿，再進同一個聊天模型。兩條路共用對話記錄，因此你先打字說「我不吃辣」，下一次用語音問晚餐，這個限制仍應留在上下文裡。

前置是[12.13聽寫與回答的分工](https://birdhackor.github.io/tiny-perceptron-vlm/12.13.html)與[7.1對話表示](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)。這條路會失去聲調、情緒等逐字稿沒有寫出的資訊，限制與[12.11](https://birdhackor.github.io/tiny-perceptron-vlm/12.11.html)相同。直接把聲音特徵送進語言模型，是另一條可學的路；本次實用成品採兩段式分工，沒有宣稱其聊天核心直接理解原始波形，也不提供朗讀回答的TTS。

圖中的「圖文聊天底座」是已學過大量圖片與文字的聊天模型；「本課微調」是在它的起始權重上調整部分數字。微調是與底座比較的候選配置；第20章依驗證保留原底座，沒有開啟這份修正。照片也能送給這位助理，提供問題需要的畫面線索。這裡先看兩種提問方式如何會合；[20.1](https://birdhackor.github.io/tiny-perceptron-vlm/20.1.html)完整介紹照片、打字與說話的共同入口。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_shared_chat.svg")))

In [ ]:
from tiny_perceptron.natural_concepts import speech_stages

report = speech_stages("我不吃辣", "我不吃拉", "選清淡的湯麵。", "選清淡的湯麵。")
print("聽寫字元錯誤率", report["transcription"]["cer"])
print("兩路回答完全相同", report["answers_identical"])
print("相同是否足以證明正確", report["answers_correct"])

`speech_stages()`的四個字串依次是正確原話、辨識逐字稿、打字路線的回答、語音路線的回答。這些文字都是我們手寫的例子，程式沒有播放音訊、載入模型或生成晚餐建議。`report`保存兩段評估；`report["transcription"]["cer"]`先取聽寫比較，再取其中的CER。

CER是字元錯誤率：把辨識結果改成正確原話最少需幾次插入、刪除或替換，再除以正確原話的字數。本例四字原話有一字替換，第一行是1／4＝0.25；兩個答案相同，第二行是True。第三行是None，這是Python表示此項結果尚未提供的值。本例尚未判定回答是否正確，不能只靠答案相同便替我們作判斷。反過來，兩條路用了不同說法，也不一定代表意思不同。要檢查回答是否合理，仍需先訂好這個聊天任務的判準。

實測要同時留下正確原話、辨識出的逐字稿、文字路徑的回答、語音路徑的回答與對話歷史。語音路徑只能收到模型辨識的逐字稿，不能偷偷替換成正確原話。這像檢查接力賽：第一棒交錯紙條與第二棒看對紙條卻答錯，需要分開找原因。[20.1](https://birdhackor.github.io/tiny-perceptron-vlm/20.1.html)先展示整套輸入分工，[20.12](https://birdhackor.github.io/tiny-perceptron-vlm/20.12.html)教你分開檢查兩站，最後成績見[20.13](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)。

練習為「我不吃辣」接著「給我晚餐建議」設計兩輪對話，指出語音入口應保留哪段歷史。這次希望能進行聊天，不要求教學模型達到長篇推理或強大考試能力。
